In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from fill_missing_mode import fill_with_equipment_mode
from assign_set_temp import assign_set_temp

In [2]:
# Load rooms
rooms = pd.read_excel(config.CLEANING_WORKBOOKS / "rooms_groups_cleaning_workbook.xlsx",
                      sheet_name="rooms"
)

In [3]:
# Load room-coordinate bridge
GIS_data = pd.read_csv(config.LABS_LIST / "UZH_arcgis_rooms.csv",
                       keep_default_na=False,  # Keep "None" as a string, not NaN
                       na_values=[""], # Only treat empty strings as NaN
                       usecols=["Raumcode", "Institut", "Arealname", "X", "Y", "x2", "y2"]
)

In [4]:
# Make each room its own entry
entries = rooms["cleaned_value"].str.split(",") #make each room its own string
entries = entries.explode() #separate list of strings into different rows
entries = entries.str.strip() #remove white space from list making
entries = entries.dropna() #remove empty room information
# entries = entries.str.split("-") #split into a list with building, floor, room

# Convert to data fram
entries = pd.DataFrame(entries)

In [5]:
# Add coordinate data to rooms
room_locations = entries.join(
    other=GIS_data.set_index("Raumcode"), # Variable that matches data string structure
    on="cleaned_value",
    how="left" #use index of our data
)

In [6]:
room_locations.to_csv(config.LABS_LIST / "study_room_coordinates.csv", index = False)